In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.table("airbnb_analytics.bronze.airbnb_listings")

df.show(10)

### Null Check

In [0]:
null_df = df.select([
    count(
        when(
            col(column).isNull() | (trim(col(column).cast("string")) == ""),
            column
        )
    ).alias(column)
    for column in df.columns
])

display(null_df)

### Remove Duplicate Records

In [0]:
df = df.dropDuplicates()

print(f"Records After Removing Duplicates : {df.count()}")

### Remove Leading & Trailing Spaces

In [0]:
string_columns = [field.name for field in df.schema.fields if field.dataType.simpleString() == "string"]

for column in string_columns:
    df = df.withColumn(column, trim(col(column)))

### Standardize Column Names

In [0]:
for column in df.columns:
    df = df.withColumnRenamed(column, column.lower())

In [0]:
display(df)

### Standaradize Text Values

In [0]:
from pyspark.sql.functions import initcap, lower, col

text_columns = [
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for column in text_columns:
    df = df.withColumn(column, initcap(lower(col(column))))

display(df)

### Convert Data Types

In [0]:
from pyspark.sql.functions import expr

df = (
    df.withColumn("id", expr("try_cast(id as BIGINT)"))
      .withColumn("host_id", expr("try_cast(host_id as BIGINT)"))
      .withColumn("latitude", expr("try_cast(latitude as DOUBLE)"))
      .withColumn("longitude", expr("try_cast(longitude as DOUBLE)"))
      .withColumn("price", expr("try_cast(price as DOUBLE)"))
      .withColumn("minimum_nights", expr("try_cast(minimum_nights as INT)"))
      .withColumn("number_of_reviews", expr("try_cast(number_of_reviews as INT)"))
      .withColumn("reviews_per_month", expr("try_cast(reviews_per_month as DOUBLE)"))
      .withColumn("calculated_host_listings_count", expr("try_cast(calculated_host_listings_count as INT)"))
)

### Verify Schema

In [0]:
df.printSchema()

### Business Validation

In [0]:
df = df.filter(
    col("id").isNotNull() &
    col("host_id").isNotNull() &
    col("room_type").isNotNull()
)

### Count Check

In [0]:
print(f"Records After Business Validation : {df.count()}")

In [0]:
display(df)

### Write to Silver

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airbnb_analytics.silver.airbnb_listings")

### Verify

In [0]:
display(spark.table("airbnb_analytics.silver.airbnb_listings"))